In [20]:
import pandas as pd
import numpy as np
import pickle
from google.colab import drive
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from pyproj import Transformer
import plotly.express as px
import matplotlib.pyplot as plt
import json as _json

## 1. Préparation des données

In [21]:
drive.mount('/content/drive')

BASE = '/content/drive/MyDrive/Projet Bigdata IA WEB/'
df = pd.read_csv(BASE + 'Data_Arbre_Cleaned.csv')
print(df.shape)
df.head()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
(9551, 19)


,X,Y,src_geo,clc_quartier,clc_secteur,id_arbre,haut_tot,haut_tronc,tronc_diam,fk_arb_etat,fk_stadedev,fk_situation,dte_plantation,age_estim,clc_nbr_diag,dte_abattage,nomfrancais,feuillage,remarquable
0,1.720058e+06,8.295025e+06,Orthophoto,Quartier saint-jean,Accueil de loisirs kergomard,1,16.0,4.0,148,En place,Adulte,Groupe,NaN,50,0.0,NaN,Tiltom,Feuillu,Non
1,1.719483e+06,8.295265e+06,Orthophoto,Quartier du vermandois,Ancienne ecole david et maigret,1,5.0,2.0,49,En place,Jeune,Isolé,NaN,20,0.0,NaN,Amelam,Feuillu,Non
2,1.722294e+06,8.295055e+06,Orthophoto,Quartier remicourt,Auberge de jeunesse,1,25.0,2.0,250,Supprimé,Adulte,Alignement,NaN,50,1.0,NaN,Popnigita,Feuillu,Non
3,1.718429e+06,8.295551e+06,Orthophoto,Quartier du vermandois,Avenue archimède,1,7.0,0.0,80,En place,Jeune,Alignement,NaN,15,NaN,NaN,Fagsylfas,Feuillu,Non
4,1.722092e+06,8.294788e+06,Orthophoto,Quartier remicourt,Avenue aristide briand,1,5.0,2.0,25,En place,Jeune,Alignement,NaN,10,0.0,NaN,Amelam,Feuillu,Non


In [22]:
cols_selected = [ 'X', 'Y', 'haut_tot', 'tronc_diam']

df_sel = df[cols_selected].dropna(subset=['X', 'Y', 'haut_tot', 'tronc_diam']).copy()
print("Shape :", df_sel.shape)
df_sel.head()

Shape : (9551, 4)


,X,Y,haut_tot,tronc_diam
0,1.720058e+06,8.295025e+06,16.0,148
1,1.719483e+06,8.295265e+06,5.0,49
2,1.722294e+06,8.295055e+06,25.0,250
3,1.718429e+06,8.295551e+06,7.0,80
4,1.722092e+06,8.294788e+06,5.0,25


##2. Apprentissage non-supervisé

In [23]:
# Entrée que l'on utilise pour effectuer le tri par taille d'arbre
features = ['haut_tot', 'tronc_diam']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_sel[features])

with open(BASE + 'scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

print("Shape X_scaled :", X_scaled.shape)

Shape X_scaled : (9551, 2)


In [24]:

algos_def = {
    'KMeans' : lambda k: KMeans(n_clusters=k, random_state=42, n_init=10),
    'AgglomerativeClustering' : lambda k: AgglomerativeClustering(n_clusters=k),
}

labels_par_algo = {}

for k in [2, 3]:
    labels_par_algo[k] = {}
    for nom, algo_fn in algos_def.items():
        labels_par_algo[k][nom] = algo_fn(k).fit_predict(X_scaled)
        print(f"k={k} | {nom} — entraîné")

k=2 | KMeans — entraîné
k=2 | AgglomerativeClustering — entraîné
k=3 | KMeans — entraîné
k=3 | AgglomerativeClustering — entraîné


## 3. Métriques

In [25]:
resultats = []

for k, algos in labels_par_algo.items():
    for nom, lbl in algos.items():
        resultats.append({
            'k'                 : k,
            'algorithme'        : nom,
            'silhouette'        : silhouette_score(X_scaled, lbl),
            'davies_bouldin'    : davies_bouldin_score(X_scaled, lbl),
            'calinski_harabasz' : calinski_harabasz_score(X_scaled, lbl),
        })

df_comp = (pd.DataFrame(resultats)
             .set_index(['k', 'algorithme'])
             .round(4))

for k in [2, 3]:
    sub = df_comp.loc[k]
    print("=" * 65)
    print(f"  k={k}")
    print("=" * 65)
    print(sub.to_string())
    print()
    print(f"  → Silhouette (↑ max)       : {sub['silhouette'].idxmax()}")
    print(f"  → Davies-Bouldin (↓ min)   : {sub['davies_bouldin'].idxmin()}")
    print(f"  → Calinski-Harabasz (↑ max): {sub['calinski_harabasz'].idxmax()}")
    print()

  k=2
                         silhouette  davies_bouldin  calinski_harabasz
algorithme                                                            
KMeans                       0.4882          0.7842         11514.2163
AgglomerativeClustering      0.4285          0.8483          9518.7964

  → Silhouette (↑ max)       : KMeans
  → Davies-Bouldin (↓ min)   : KMeans
  → Calinski-Harabasz (↑ max): KMeans

  k=3
                         silhouette  davies_bouldin  calinski_harabasz
algorithme                                                            
KMeans                       0.3946          0.8735         10650.6743
AgglomerativeClustering      0.3922          0.9224          8438.3544

  → Silhouette (↑ max)       : KMeans
  → Davies-Bouldin (↓ min)   : KMeans
  → Calinski-Harabasz (↑ max): KMeans



In [26]:
# Sélection du meilleur algorithme par k

# Le nombre de métrique qui ont un meilleur score
def meilleur_algo(sub):
    votes = {}
    for nom in sub.index:
        votes[nom] = 0
    votes[sub['silhouette'].idxmax()] += 1
    votes[sub['davies_bouldin'].idxmin()] += 1
    votes[sub['calinski_harabasz'].idxmax()] += 1
    return max(votes, key=votes.get)

models = {}

for k in [2, 3]:
    sub = df_comp.loc[k]
    best_nom = meilleur_algo(sub)
    best_lbl = labels_par_algo[k][best_nom]
    col = f'cluster_k{k}'
    cat_col  = f'categorie_k{k}'

    df_sel[col] = best_lbl

    ordre = df_sel.groupby(col)['haut_tot'].mean().sort_values().index.tolist()
    lbl_map = (
        {ordre[0]: 'Petit', ordre[1]: 'Grand'}
        if k == 2
        else {ordre[0]: 'Petit', ordre[1]: 'Moyen', ordre[2]: 'Grand'}
    )
    df_sel[cat_col] = df_sel[col].map(lbl_map)
    models[k] = {'model': best_nom, 'label_map': lbl_map}

    print(f"\n── k={k}  →  algorithme retenu : {best_nom} ──────────────────")
    print(df_sel.groupby(cat_col)[features].mean().round(1))

# Sauvegarde en .pkl et .json
for k, m in models.items():
    best_fitted = algos_def[m['model']](k)
    best_fitted.fit(X_scaled)
    with open(BASE + f'model_k{k}.pkl', 'wb') as f:
        pickle.dump(best_fitted, f)

    data = {
        'algorithme' : m['model'],
        'label_map'  : {int(key): val for key, val in m['label_map'].items()},
        'features'   : features,
    }
    with open(BASE + f'model_k{k}.json', 'w', encoding='utf-8') as f:
        _json.dump(data, f, ensure_ascii=False, indent=2)

    print(f" k={k} → model_k{k}.pkl + model_k{k}.json  ({m['model']})")


── k=2  →  algorithme retenu : KMeans ──────────────────
              haut_tot  tronc_diam
categorie_k2                      
Grand             17.1       156.2
Petit              7.7        74.4

── k=3  →  algorithme retenu : KMeans ──────────────────
              haut_tot  tronc_diam
categorie_k3                      
Grand             19.6       171.7
Moyen             10.7       115.7
Petit              6.3        49.3
 k=2 → model_k2.pkl + model_k2.json  (KMeans)
 k=3 → model_k3.pkl + model_k3.json  (KMeans)


## 4. Visualiation sur une carte

In [27]:

# Transformer les coordonnées dans le bon format

transformer = Transformer.from_crs("EPSG:3949", "EPSG:4326", always_xy=True)
df_sel['lon'], df_sel['lat'] = transformer.transform(
    df_sel['X'].values, df_sel['Y'].values
)
df_sel[['X', 'Y', 'lon', 'lat']].head()

,X,Y,lon,lat
0,1.720058e+06,8.295025e+06,3.278914,49.854112
1,1.719483e+06,8.295265e+06,3.270928,49.856292
2,1.722294e+06,8.295055e+06,3.310014,49.854309
3,1.718429e+06,8.295551e+06,3.256283,49.858897
4,1.722092e+06,8.294788e+06,3.307180,49.851914


In [29]:

#Création de la carte en html

cols_export = ['lat', 'lon', 'haut_tot',
               'tronc_diam', 'categorie_k2', 'categorie_k3']

records = (df_sel[cols_export]
           .rename(columns={'categorie_k2':'cat_k2', 'categorie_k3':'cat_k3'}))

data_json = records.to_json(orient='records', force_ascii=False)

lat_c = round(float(df_sel['lat'].median()), 4)
lon_c = round(float(df_sel['lon'].median()), 4)

HTML_TEMPLATE = (
'<!DOCTYPE html>\n'
'<html lang="fr">\n'
'<head>\n'
'  <meta charset="UTF-8">\n'
'  <meta name="referrer" content="strict-origin-when-cross-origin">\n'
'  <title>Carte des arbres</title>\n'
'  <script src="https://cdn.plot.ly/plotly-2.27.0.min.js"></script>\n'
'  <style>\n'
'    * { box-sizing: border-box; margin: 0; padding: 0; }\n'
'    body { font-family: Arial, sans-serif; display: flex; flex-direction: column; height: 100vh; }\n'
'    #controls { display: flex; align-items: center; gap: 24px; padding: 10px 16px; background: #fff; border-bottom: 1px solid #ddd; flex-wrap: wrap; }\n'
'    #controls span { font-size: .8rem; color: #555; font-weight: 600; }\n'
'    .toggle-btns button { padding: 5px 16px; border: 1px solid #333; background: #fff; border-radius: 4px; cursor: pointer; font-size: .82rem; }\n'
'    .toggle-btns button.active { background: #333; color: #fff; }\n'
'    .cat-checks { display: flex; gap: 12px; }\n'
'    .cat-checks label { display: flex; align-items: center; gap: 5px; font-size: .82rem; cursor: pointer; }\n'
'    .dot { width: 10px; height: 10px; border-radius: 50%; }\n'
'    #map { flex: 1; }\n'
'  </style>\n'
'</head>\n'
'<body>\n'
'<div id="controls">\n'
'  <div>\n'
'    <span>Clustering &nbsp;</span>\n'
'    <span class="toggle-btns">\n'
'      <button id="btn-k2" onclick="setK(2)">K = 2</button>\n'
'      <button id="btn-k3" onclick="setK(3)" class="active">K = 3</button>\n'
'    </span>\n'
'  </div>\n'
'  <div style="display:flex;align-items:center;gap:8px">\n'
'    <span>Cat\u00e9gories &nbsp;</span>\n'
'    <div class="cat-checks" id="cat-checks"></div>\n'
'  </div>\n'
'</div>\n'
'<div id="map"></div>\n'
'<script>\n'
'const DATA   = __DATA__;\n'
'const LAT_C  = __LATC__;\n'
'const LON_C  = __LONC__;\n'
'const COLORS = { Petit:"#2196F3", Moyen:"#FF9800", Grand:"#F44336" };\n'
'const ORDER  = ["Petit","Moyen","Grand"];\n'
'let currentK   = 3;\n'
'let activeCats = new Set(["Petit","Moyen","Grand"]);\n'
'\n'
'function catsForK(k){ return k===2 ? ["Petit","Grand"] : ["Petit","Moyen","Grand"]; }\n'
'\n'
'function setK(k){\n'
'  currentK = k;\n'
'  activeCats = new Set(catsForK(k));\n'
'  document.getElementById("btn-k2").classList.toggle("active", k===2);\n'
'  document.getElementById("btn-k3").classList.toggle("active", k===3);\n'
'  renderChecks();\n'
'  renderMap();\n'
'}\n'
'\n'
'function renderChecks(){\n'
'  const c = document.getElementById("cat-checks");\n'
'  c.innerHTML = "";\n'
'  catsForK(currentK).forEach(cat => {\n'
'    const l = document.createElement("label");\n'
'    l.innerHTML =\n'
'      "<input type=\\"checkbox\\" "+(activeCats.has(cat)?"checked":"")+\n'
'      " onchange=\\"toggleCat(\'"+cat+"\',this.checked)\\">"+\n'
'      "<span class=\\"dot\\" style=\\"background:"+COLORS[cat]+"\\"></span>"+cat;\n'
'    c.appendChild(l);\n'
'  });\n'
'}\n'
'\n'
'function toggleCat(cat, checked){\n'
'  if(checked) activeCats.add(cat); else activeCats.delete(cat);\n'
'  renderMap();\n'
'}\n'
'\n'
'function renderMap(){\n'
'  const gd    = document.getElementById("map");\n'
'  const field = currentK === 2 ? "cat_k2" : "cat_k3";\n'
'  const cats  = [...activeCats].sort((a,b) => ORDER.indexOf(a)-ORDER.indexOf(b));\n'
'\n'
'  const mb     = gd._fullLayout && gd._fullLayout.mapbox;\n'
'  const zoom   = mb ? mb.zoom   : 11;\n'
'  const center = mb ? mb.center : { lat: LAT_C, lon: LON_C };\n'
'\n'
'  const traces = cats.map(cat => {\n'
'    const pts = DATA.filter(d => d[field] === cat);\n'
'    return {\n'
'      type:"scattermapbox", lat:pts.map(d=>d.lat), lon:pts.map(d=>d.lon),\n'
'      mode:"markers", name:cat,\n'
'      marker:{ color:COLORS[cat], size:6, opacity:0.78 },\n'
#'      text:pts.map(d=>"<b>"+d.nom+"</b><br>Quartier : "+d.quartier+"<br>Hauteur : "+d.haut_tot+" m"),\n'
'      text:pts.map(d=>"Hauteur : "+d.haut_tot+" m"),\n'

'      hovertemplate:"%{text}<extra></extra>",\n'
'    };\n'
'  });\n'
'\n'
'  Plotly.react("map", traces, {\n'
'    mapbox:{ style:"open-street-map", center:center, zoom:zoom },\n'
'    showlegend: false,\n'
'    margin:{ t:0, b:0, l:0, r:0 },\n'
'    paper_bgcolor:"transparent",\n'
'  }, { responsive:true });\n'
'}\n'
'\n'
'renderChecks(); renderMap();\n'
'</script>\n'
'</body>\n'
'</html>'
)

html_final = (HTML_TEMPLATE
              .replace('__DATA__', data_json)
              .replace('__LATC__',  str(lat_c))
              .replace('__LONC__',  str(lon_c)))

out_path = BASE + 'carte_arbres_interactive.html'
with open(out_path, 'w', encoding='utf-8') as f:
    f.write(html_final)

print(f"HTML : {out_path}  ({len(records):,} arbres)")

HTML : /content/drive/MyDrive/Projet Bigdata IA WEB/carte_arbres_interactive.html  (9,551 arbres)
